In [1]:
from dotenv import load_dotenv

load_dotenv()

import asyncio
import pandas as pd

def async_run(coro):
    """Evite le probleme: event loop is closed."""
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    
    # Exécute la coroutine jusqu'à ce qu'elle soit terminée
    return loop.run_until_complete(coro)

dataset = pd.read_parquet("train_gold_relevant.parquet")
dataset.head()

,question,response,action,embedding
0,De : communicat@arch.be <communicat@arch.be> \...,"Date: Fri, 05 Jul 2024 10:24:39 +0000\nTo: pat...",REDIRECT,"[-0.00039283143, 0.010335003, -0.004006752, 0...."
1,De : communicat@arch.be <communicat@arch.be> \...,"Date: Fri, 05 Jul 2024 10:24:39 +0000\nTo: pat...",REDIRECT,"[-0.00039283143, 0.010335003, -0.004006752, 0...."
16,De : communicat@arch.be <communicat@arch.be> \...,"Date: Mon, 01 Jul 2024 13:59:19 +0000\nTo: ""Ar...",HUMAN_TAKEOVER,"[0.0014402776, 0.01795132, -0.0048872265, 0.00..."
17,De : communicat@arch.be <communicat@arch.be> \...,"Date: Mon, 01 Jul 2024 13:59:19 +0000\nTo: ""Ar...",HUMAN_TAKEOVER,"[0.0014402776, 0.01795132, -0.0048872265, 0.00..."
18,**Van:** _[communicat@arch.be](mailto:communi...,**Van:** digita <[digita@arch.be](mailto:digi...,HUMAN_TAKEOVER,"[0.00882528, 0.0057901046, -0.005896127, 0.002..."


In [2]:
from langgraph.graph import (
    END,
    START,
    StateGraph,
)

from agents.local_llm.agatha.schemas import (
    AgentState,
)
from agents.local_llm.agatha.steps import (
    cond_enough_info,
    cond_what_tool,
    step_formalize,
    step_initialize,
    step_set_objective,
    tool_online_help,
)

async def step_select_tool(state: AgentState) -> AgentState:
    name = await cond_what_tool(state)
    return {"call_tool": name} # type: ignore

async def pseudo_rsp(state: AgentState) -> AgentState:
    return {"call_tool": "Respond"} # type: ignore

AGATHA_TOOL_SELECTION = (
    StateGraph(AgentState)
        .add_edge(START, "Initialize")
        #
        .add_node("Initialize", step_initialize)
        .add_node("Set Objective", step_set_objective)
        .add_node("Formalize",  step_formalize)
        .add_node("Select Tool", step_select_tool)
        .add_node("Respond", pseudo_rsp)
        #
        .add_conditional_edges("Initialize", cond_enough_info)
        .add_edge("Set Objective", "Formalize")
        .add_edge("Formalize", "Select Tool")
        .add_edge("Select Tool", END)
        .add_edge("Respond", END)
        .compile()
)

In [19]:
def smallbatch(ds, n): 
    rs = []
    for x in ds.iterrows():
        rs.append(x)
        if len(rs) >= n:
            yield rs
            rs = []
    if rs: 
        yield rs



In [23]:
rs = []
for b in smallbatch(dataset, 4):
    rs.append(await AGATHA_TOOL_SELECTION.abatch([
        {'input': q["question"]} for i,q in b
    ]))

KeyError: '{"ranking": ["Analyses of Records / Free Search", "Analyses of Records / Search Person", "Online Help Resources", "Genealogical sources / Church Registers", "Genealogical sources / Civil State Registers"]}'

In [24]:
rs

[[{'messages': [SystemMessage(content='\nYour name is Arkey. \n\nYou are an expert AI customer support at the Belgian State Archive. \nYou are a specialist who has extensively been working with IT on the\ndevelopment of AGATHA (www.agatha.arch.be) the web portal to search the\nBelgian State Archive. You are quite efficient at working with the various\nsearch engines offered on the AGATHA platform.\n\nYou are known for being factual, and always providing reliable information. \nWhenever you doubt, about a statement you wish to make (e.g. if you \nhaven\'t had the opportunity to verify the data yourself), \nyou prefer to be transparent about it rather than risking to give \nerroneous information.\n\n**RESOURCES**\nIn order to achieve your goals, you can use the following resources:\n\n    * \'Online Help Resources\' that will let you search in \n        the user manuals and FAQ sections of the AGATHA platform. \n        This is mostly useful to answer *HOW DO I ...* questions.\n\n    * \

In [25]:
len(rs)

58

In [27]:
len(rs[0])

4

In [28]:
r = [x for q in rs for x in q]
len(r)

232

In [30]:
part = dataset[:232]
part["run"] = r

/tmp/ipykernel_44807/2371713214.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  part["run"] = r


In [31]:
part

,question,response,action,embedding,run
0,De : communicat@arch.be <communicat@arch.be> \...,"Date: Fri, 05 Jul 2024 10:24:39 +0000\nTo: pat...",REDIRECT,"[-0.00039283143, 0.010335003, -0.004006752, 0....",{'messages': [content='\nYour name is Arkey. \...
1,De : communicat@arch.be <communicat@arch.be> \...,"Date: Fri, 05 Jul 2024 10:24:39 +0000\nTo: pat...",REDIRECT,"[-0.00039283143, 0.010335003, -0.004006752, 0....",{'messages': [content='\nYour name is Arkey. \...
16,De : communicat@arch.be <communicat@arch.be> \...,"Date: Mon, 01 Jul 2024 13:59:19 +0000\nTo: ""Ar...",HUMAN_TAKEOVER,"[0.0014402776, 0.01795132, -0.0048872265, 0.00...",{'messages': [content='\nYour name is Arkey. \...
17,De : communicat@arch.be <communicat@arch.be> \...,"Date: Mon, 01 Jul 2024 13:59:19 +0000\nTo: ""Ar...",HUMAN_TAKEOVER,"[0.0014402776, 0.01795132, -0.0048872265, 0.00...",{'messages': [content='\nYour name is Arkey. \...
18,**Van:** _[communicat@arch.be](mailto:communi...,**Van:** digita <[digita@arch.be](mailto:digi...,HUMAN_TAKEOVER,"[0.00882528, 0.0057901046, -0.005896127, 0.002...",{'messages': [content='\nYour name is Arkey. \...
...,...,...,...,...,...
358,**De :** Antoine Olivier <Olivier.Antoine@arc...,"Date: Thu, 27 Jun 2024 15:14:10 +0000\nTo: ""An...",ACTUAL_SEARCH,"[0.0029714052, 0.010901725, -0.0128101185, -0....",{'messages': [content='\nYour name is Arkey. \...
359,**De :** Wouters Marie-Therese <M.T.Wouters@s...,"Date: Fri, 21 Jun 2024 11:24:45 +0000\nTo: M.T...",ACTUAL_SEARCH,"[0.0026952175, 0.00851775, -0.008765841, 0.005...",{'messages': [content='\nYour name is Arkey. \...
360,**De :** Van der Auwera Jan <jan.vanderauwera...,"Date: Thu, 27 Jun 2024 11:41:40 +0000\nTo: jan...",ACTUAL_SEARCH,"[0.0020539428, 0.0063678296, -0.007925288, 0.0...",{'messages': [content='\nYour name is Arkey. \...
361,**De :** Mertens Pierre <pierre.mertens@skyne...,"Date: Fri, 05 Jul 2024 13:25:15 +0000\nTo: pie...",ACTUAL_SEARCH,"[0.0039936663, 0.009499178, -0.0086728195, 0.0...",{'messages': [content='\nYour name is Arkey. \...


In [34]:
run = pd.DataFrame(r)
run["call_tool"] = run["call_tool"].astype("category")

In [41]:
run.head()

,messages,input,thought,criteria,call_tool,n
0,[content='\nYour name is Arkey. \n\nYou are an...,De : communicat@arch.be <communicat@arch.be> \...,I want to find the details of Van Muylder Léon...,municipality_name_in_french_or_dutch='Molenbee...,Analyses of Records / Search Person,1.0
1,[content='\nYour name is Arkey. \n\nYou are an...,De : communicat@arch.be <communicat@arch.be> \...,I want to find the details of Van Muylder Léon...,municipality_name_in_french_or_dutch='Molenbee...,Analyses of Records / Free Search,1.0
2,[content='\nYour name is Arkey. \n\nYou are an...,De : communicat@arch.be <communicat@arch.be> \...,I want to find the details of a death that hap...,municipality_name_in_french_or_dutch='Saint-Hu...,Online Help Resources,1.0
3,[content='\nYour name is Arkey. \n\nYou are an...,De : communicat@arch.be <communicat@arch.be> \...,I want to find the details of a death where th...,municipality_name_in_french_or_dutch='Saint-Hu...,Genealogical sources / Civil State Registers,1.0
4,[content='\nYour name is Arkey. \n\nYou are an...,**Van:** _[communicat@arch.be](mailto:communi...,"I want to find the help section about ""soldate...",municipality_name_in_french_or_dutch=None pari...,Online Help Resources,1.0


In [52]:
def conversation(state): 
    rsp = ""
    for m in state["messages"]:
        rsp += f"<|{m.type}|>\n{m.content}\n\n"

    rsp += f"<|objective|>\n{state["thought"]}\n\n"
    return rsp


df = pd.DataFrame()
df["prompt"] = run.apply(conversation, axis=1)
df["tool"]   = run["call_tool"]

df.to_parquet("tool_classifier_dataset.parquet")

In [53]:
df.to_csv("tool_classifier.csv")

In [85]:
import json
from textwrap import dedent

def simplified(state): 
    crit = state["criteria"].model_dump_json(indent=4) if hasattr(state["criteria"], "model_dump_json") else ""
    obj  = state["thought"]

    return dedent(
f"""
Target: {obj}
Facts: 
{crit}
"""
    ).strip()

df2 = pd.DataFrame()
df2["input"] = run["input"]
df2["prompt"] = run.apply(simplified, axis=1)
df2["tool"]   = run["call_tool"]

df2.to_csv("tool_classifier.csv")


In [81]:
df2

,prompt,tool
0,Target: I want to find the details of Van Muyl...,Analyses of Records / Search Person
1,Target: I want to find the details of Van Muyl...,Analyses of Records / Free Search
2,Target: I want to find the details of a death ...,Online Help Resources
3,Target: I want to find the details of a death ...,Genealogical sources / Civil State Registers
4,Target: I want to find the help section about ...,Online Help Resources
...,...,...
227,Target: I want to find the details of dossier ...,Analyses of Records / Free Search
228,Target: I want to find the details of the birt...,Analyses of Records / Free Search
229,Target: I want to find the details of an act o...,Analyses of Records / Free Search
230,Target: I want to find the details of a marria...,Analyses of Records / Free Search


In [1]:
import pandas as pd

df3 = pd.read_csv("tool_classifier.csv")
df3 = df3[["input", "prompt", "tool"]]

df3["prompt"].nunique()

178

In [6]:
df3.groupby("prompt").min().reset_index().to_csv("tool_classifier.csv", encoding="utf8", index=False)